In [ ]:
import arcpy

# --- Portal bilgileri ---
portal_url = "https://geo.ipa.istanbul/portal/"
username = "kullanici.adi"
password = "kullanici.sifresi"

# --- Portal'a giriş ---
arcpy.SignInToPortal(portal_url, username, password)

# --- Feature Service URL (REST, item linki değil) ---
feature_service_url = "https://geo.ipa.istanbul/server/rest/services/Hosted/detayli_ak/FeatureServer/0"

### --- Feature Layer oluştur ---
layer_name = "detayli_ak_lyr"
arcpy.management.MakeFeatureLayer(feature_service_url, layer_name)

# --- Basit kontrol ---
print("Portal:", arcpy.GetActivePortalURL())
print("Layer var mı?:", arcpy.Exists(layer_name))
print("Kayıt sayısı:", arcpy.management.GetCount(layer_name)[0])


In [ ]:
import csv
from collections import Counter

# --- Katmanlar ---
source_layer = "detayli_ak_lyr"
filtered_layer = "detayli_ak_filtered"

# --- CSV yolu ---
csv_path = r"csv_yolu"

# --- CSV'den domain sözlüğü (kod -> açıklama) ---
domain_lookup = {}

with open(csv_path, newline='', encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        domain_lookup[str(row["Kod"])] = row["Adi"]

# --- Anlamlı veri filtresi ---
where_clause = "alt_kullanim IS NOT NULL"

arcpy.management.MakeFeatureLayer(
    source_layer,
    filtered_layer,
    where_clause
)

# --- Toplam kayıt ---
total = int(arcpy.management.GetCount(filtered_layer)[0])

# --- Frekans analizi ---
values = []
with arcpy.da.SearchCursor(filtered_layer, ["alt_kullanim"]) as cursor:
    for row in cursor:
        values.append(str(row[0]))  # kodlar string'e çevrildi

freq = Counter(values)

# --- ÇIKTI ---
print("ANLAMLI KAYIT SAYISI:", total)
print("\nALT_KULLANIM DAĞILIMI (açıklama / adet / %):\n")

for kod, adet in freq.most_common():
    aciklama = domain_lookup.get(kod, f"Bilinmeyen ({kod})")
    print(f"{aciklama} : {adet}  |  %{(adet/total)*100:.2f}")


In [ ]:
import os
# Kaynak layer (zaten hazır)
source_layer = "detayli_ak_filtered"

# GDB'nin oluşturulacağı klasör
workspace_folder = r"C:\Users\____.korucu\Desktop\veri_temin_____"

# GDB adı
gdb_name = "detayli_ak_local.gdb"
gdb_path = os.path.join(workspace_folder, gdb_name)

# Eğer GDB yoksa oluştur
if not arcpy.Exists(gdb_path):
    arcpy.management.CreateFileGDB(workspace_folder, gdb_name)

# Feature class adı
output_fc = os.path.join(gdb_path, "detayli_ak_filtered")

# Feature class olarak kopyala
arcpy.management.CopyFeatures(source_layer, output_fc)

print("Lokal GDB oluşturuldu:", gdb_path)
print("Feature class yazıldı:", output_fc)
print("Kayıt sayısı:", arcpy.management.GetCount(output_fc)[0])